In [1]:
import optuna
import torch
import os
import numpy as np
from solver import Solver
from data.data_loader import get_loader
from utils.genutils import write_print, mkdir

# Import your project configuration
import argparse

# Create a temporary configuration object
class Config:
    def __init__(self):
        self.lr = 0.001
        self.momentum = 0.9
        self.batch_size = 32
        self.num_epochs = 220
        self.dataset = 'tomatod'
        self.new_size = 300
        self.model = 'SSD'
        self.weight_decay = 0.0005
        self.learning_sched = [160, 190]
        self.use_gpu = torch.cuda.is_available()
        self.model_save_path = "./weights"
        self.model_test_path = "./tests"
        self.model_eval_path = "./eval"
        self.means = (104, 117, 123)  # Default values for normalization
        self.mode = "train"
        self.tomatod_data_path = "./data/Datasets/Tomatod/"
        self.class_count = 4
        self.anchor_config = 'SSD-300'  # Default anchor configuration
        self.scale_initial = 0.1
        self.scale_min = 0.2
        self.scale_max = 1.05

# Create a global config object
config = Config()

In [2]:
class ConfigObject:
    """Converts a dictionary into an object with attributes."""
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

In [10]:
def objective(trial):
    """Objective function for Optuna hyperparameter optimization"""

    # Step 1: Define the hyperparameters to tune with Optuna
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)  # Learning Rate (Log Scale)
    momentum = trial.suggest_float('momentum', 0.85, 0.99)  # Momentum
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])  # Batch Size
    num_epochs = trial.suggest_int('num_epochs', 50, 150, step=10)  # Number of epochs

    # Step 2: Define a complete config dictionary (NO missing keys)
    config_dict = {
        # Dataset information
        'input_channels': 3,
        'class_count': 4,
        'dataset': 'tomatod',  # Default dataset, update dynamically for Optuna
        'new_size': 300,
        'means': (104, 117, 123),
        'anchor_config': 'SSD-300',
        'scale_initial': 0.1,
        'scale_min': 0.2,
        'scale_max': 1.05,

        # Training settings
        'lr': lr,  # Tuned by Optuna
        'momentum': momentum,  # Tuned by Optuna
        'weight_decay': 0.0005,
        'num_epochs': num_epochs,  # Tuned by Optuna
        'learning_sched': [160, 190],
        'warmup_epoch': 0,
        'sched_gamma': 0.1,
        'batch_size': batch_size,  # Tuned by Optuna
        'batch_multiplier': 1,

        # Model architecture settings
        'model': 'SSD',
        'basenet': 'vgg16_reducedfc.pth',
        'resnet_model': '18',
        'densenet_model': '121',
        'resnext_model': '50_32x4d',
        'pretrained_model': None,  # Default to None, update dynamically if needed
        'coco_weights': None,

        # Loss settings
        'loss_config': 'multibox',
        'pos_neg_ratio': 3,

        # Miscellaneous settings
        'mode': 'train',
        'use_gpu': torch.cuda.is_available(),

        # Testing settings
        'max_per_image': 50,
        'score_threshold': 0.01,
        'nms_threshold': 0.5,
        'iou_threshold': 0.5,

        # Dataset paths
        'voc_config': '0712',
        'voc_data_path': '../../Datasets/PascalVOC/',
        'use_07_metric': True,
        'coco_year': '2017',
        'coco_data_path': '../../Datasets/Coco/',
        'tomatod_data_path': 'data/Datasets/Tomatod/',
        'ccrop_data_path': '../../Datasets/CCROP/',
        'camocrops_data_path': '../../Datasets/CamoCrops/',

        # Paths
        'model_save_path': './weights',
        'model_test_path': './tests',
        'model_eval_path': './eval',

        # Logging settings
        'loss_log_step': 1,
        'model_save_step': 5,
    }

    config_obj = ConfigObject(config_dict)  # Convert dictionary to object

    # Step 3: Define a unique version name for each trial
    version = f"optuna_lr{lr:.6f}_mom{momentum:.6f}_bs{batch_size}_epochs{num_epochs}"

    # Ensure logs directory exists
    log_dir = "./logs"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)

    # Ensure model save directory exists
    model_save_dir = os.path.join(config_dict['model_save_path'], version)
    if not os.path.exists(model_save_dir):
        os.makedirs(model_save_dir)

    # Define output file path for logs
    output_txt = os.path.join(log_dir, f"{version}.txt")

    config_obj = ConfigObject(config_dict)  # Convert dictionary to object
    data_loader = get_loader(config_obj)  # Load dataset with current config


    solver = Solver(version=version, data_loader=data_loader, config=config_dict, output_txt=output_txt)

    
    # Step 4: Train the model with these hyperparameters
    solver.train()

    # Step 5: Retrieve validation loss
    val_loss = get_best_validation_loss(version)
    return val_loss

In [4]:
def get_best_validation_loss(version):
    """Reads the best validation loss from the log file"""
    log_path = f'./logs/{version}.txt'
    if not os.path.exists(log_path):
        return float('inf')  # Return a high loss if no log is found
    
    best_loss = float('inf')
    with open(log_path, 'r') as f:
        for line in f:
            if "val_loss" in line:  # Assuming your logs contain val_loss
                loss = float(line.split("val_loss: ")[1])
                best_loss = min(best_loss, loss)
    
    return best_loss


In [5]:
# Create an Optuna study to minimize validation loss
study = optuna.create_study(direction='minimize')

# Run optimization for 20 trials
study.optimize(objective, n_trials=20)

[I 2025-02-21 19:04:18,631] A new study created in memory with name: no-name-47110447-ba63-4baf-b300-af7543f5d47e
C:\Users\audrea\Desktop\sk00L\CIVI\SSD FINAL\SSD-PyTorch\models\ssd.py:88: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on 

SSD(
  (base): ModuleList(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ce

100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:30<00:00,  4.29s/it]


Elapsed 0:00:30.030394/0:00:04.290056 -- 1:34:40.034492, Epoch [1/190], Iter [7/7], class_loss: 11.7759, loc_loss: 4.7218, loss: 16.4978


100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:28<00:00,  4.05s/it]


...

100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:27<00:00,  3.93s/it]


Elapsed 0:01:54.433084/0:00:04.086896 -- 1:28:45.225313, Epoch [4/190], Iter [7/7], class_loss: 6.3229, loc_loss: 3.6699, loss: 9.9928


100%|████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:27<00:00,  3.96s/it]
[W 2025-02-21 19:06:41,034] Trial 0 failed with parameters: {'lr': 9.726065813017749e-05, 'momentum': 0.9165902957062735, 'batch_size': 32, 'num_epochs': 190} because of the following error: RuntimeError('Parent directory ./weights\\optuna_lr0.000097_mom0.916590_bs32_epochs190 does not exist.').
Traceback (most recent call last):
  File "C:\Users\audrea\anaconda3\envs\thesis\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\audrea\AppData\Local\Temp\ipykernel_25388\1244615618.py", line 98, in objective
    solver.train()
  File "C:\Users\audrea\Desktop\sk00L\CIVI\SSD FINAL\SSD-PyTorch\solver.py", line 339, in train
    self.save_model(e)
  File "C:\Users\audrea\Desktop\sk00L\CIVI\SSD FINAL\SSD-PyTorch\solver.py", line 192, in save_model
    torch.save(self.model.state_dict(), path)
  File "C:\Users

Elapsed 0:02:22.125567/0:00:04.060730 -- 1:27:42.706717, Epoch [5/190], Iter [7/7], class_loss: 5.6865, loc_loss: 4.1798, loss: 9.8663


RuntimeError: Parent directory ./weights\optuna_lr0.000097_mom0.916590_bs32_epochs190 does not exist.

In [ ]:
print("Best hyperparameters:", study.best_params)
